# Cortical Morphometric Feature Extraction (VolSubGroup)

Parse FreeSurfer `lh.aparc.stats` / `rh.aparc.stats` files and extract eight cortical morphometric features (GrayVol, SurfArea, ThickAvg, ThickStd, MeanCurv, GausCurv, FoldInd, CurvInd) for each participant.

In [ ]:
import pandas as pd
import numpy as np
import os
import sys
import re
from tqdm import tqdm

# ============================================================
# Configuration
# ============================================================
EXCEL_FILE_NAME = 'data/ABIDE2_AGE.xlsx'          # subject info file
DEFAULT_PARCELLATION_NAME = 'aparc'                # Desikan-Killiany atlas

# The eight morphometric features to extract
FEATURES_TO_EXTRACT = [
    'GrayVol', 'SurfArea', 'ThickAvg', 'ThickStd',
    'MeanCurv', 'GausCurv', 'FoldInd', 'CurvInd',
]

# Map site IDs to actual directory names (fix inconsistencies)
SITE_TO_DIR_MAP = {
    'ABIDEII-ETH_1': 'ABIDEII-ETHZ_1',
    'ABIDEII-SU_2': 'ABIDEII-STANFORD',
    'ABIDEII-U_MIA_1': 'ABIDEII-UM',
}

def get_bids_site_dir(site_id):
    """Return the actual site directory name; fall back to the site ID itself."""
    return SITE_TO_DIR_MAP.get(site_id, site_id)

# ============================================================
# FreeSurfer stats file parser
# ============================================================
def parse_aparc_stats_file(file_path, hemisphere):
    """
    Parse a FreeSurfer *.aparc.stats file and extract the target morphometric
    features for every cortical region.
    """
    if not os.path.exists(file_path):
        return None, f"File not found: {file_path}"

    try:
        with open(file_path, 'r') as f:
            file_content = f.read()
    except Exception as e:
        return None, f"Read error: {e}"

    lines = file_content.split('\n')
    data_start_line = -1
    col_headers = None

    # Find the '# ColHeaders' line and the start of the data rows
    for i, line in enumerate(lines):
        stripped_line = line.strip()
        if stripped_line.startswith('# ColHeaders'):
            header_part = stripped_line.replace('# ColHeaders', '').strip()
            col_headers = re.split(r'\s+', header_part)
            data_start_line = i + 1
            break

    if data_start_line == -1 or col_headers is None:
        return None, "No '# ColHeaders' line found."

    # Map each target feature to its column index
    col_indices = {}
    for feature in FEATURES_TO_EXTRACT:
        try:
            col_indices[feature] = col_headers.index(feature)
        except ValueError:
            return None, f"Feature '{feature}' not found in the file."

    # Parse data rows
    clean_data = [line.strip() for line in lines[data_start_line:]
                  if line.strip() and not line.strip().startswith('#')]
    records = []
    for line in clean_data:
        parts = re.split(r'\s+', line.strip())
        if not parts or parts == ['']:
            continue
        try:
            struct_name = parts[0]
            region_data = {'Label': f"{hemisphere}_{struct_name}"}
            for feature, index in col_indices.items():
                if index >= len(parts):
                    raise IndexError(f"Row too short (len {len(parts)}) for index {index}")
                region_data[feature] = float(parts[index])
            records.append(region_data)
        except (IndexError, ValueError):
            continue

    if not records:
        return None, "No valid region rows extracted."
    return pd.DataFrame(records), None

def extract_aparc_stats_features(bids_root_dir, participant_id, parcellation_name):
    """
    Locate the FreeSurfer stats files for a participant and extract features.
    """
    # Locate the FreeSurfer output directory
    freesurfer_path = None
    deriv_fs_dir = os.path.join(bids_root_dir, 'derivatives', 'freesurfer')
    if os.path.isdir(deriv_fs_dir):
        try:
            fs_sub_dirs = [d for d in os.listdir(deriv_fs_dir) if d.startswith(participant_id)]
        except OSError:
            fs_sub_dirs = []
        if len(fs_sub_dirs) == 1:
            freesurfer_path = os.path.join(deriv_fs_dir, fs_sub_dirs[0])
        elif len(fs_sub_dirs) > 1:
            session_dirs = [d for d in fs_sub_dirs if '_ses-' in d]
            if len(session_dirs) == 1:
                freesurfer_path = os.path.join(deriv_fs_dir, session_dirs[0])
            else:
                return None, f"Multiple session dirs found: {fs_sub_dirs}"

    if freesurfer_path is None:
        standard_fs_path = os.path.join(bids_root_dir, 'derivatives', 'freesurfer', participant_id)
        if os.path.isdir(standard_fs_path):
            freesurfer_path = standard_fs_path

    if freesurfer_path is None:
        return None, f"FreeSurfer dir not found for {participant_id}"

    lh_file = os.path.join(freesurfer_path, 'stats', f'lh.{parcellation_name}.stats')
    rh_file = os.path.join(freesurfer_path, 'stats', f'rh.{parcellation_name}.stats')

    lh_df, lh_error = parse_aparc_stats_file(lh_file, 'lh')
    if lh_df is None:
        return None, f"LH parse failed: {lh_error}"
    rh_df, rh_error = parse_aparc_stats_file(rh_file, 'rh')
    if rh_df is None:
        return None, f"RH parse failed: {rh_error}"

    combined_df = pd.concat([lh_df, rh_df], ignore_index=True)
    combined_df.insert(0, 'participant_id', participant_id)
    return combined_df, None

# ============================================================
# Main execution
# ============================================================
if __name__ == '__main__':
    try:
        df_subjects = pd.read_excel(EXCEL_FILE_NAME)
    except Exception as e:
        print(f"Fatal: cannot load {EXCEL_FILE_NAME}: {e}")
        sys.exit(1)

    total_subjects = len(df_subjects)
    print(f"Loaded {total_subjects} participants.")
    print(f"Extracting {len(FEATURES_TO_EXTRACT)} features with atlas: {DEFAULT_PARCELLATION_NAME}")

    all_features = []
    failed_subjects = {}

    for index, row in tqdm(df_subjects.iterrows(), total=total_subjects, desc="Subjects"):
        subid_raw = str(row['SUBID'])
        site_id = row['SITE']
        site_dir = get_bids_site_dir(site_id)
        bids_root = os.path.join('/path/to/ABIDE-II', site_dir)  # <-- set local data root
        participant_id = f"sub-{subid_raw}"

        result_df, error_msg = extract_aparc_stats_features(bids_root, participant_id, DEFAULT_PARCELLATION_NAME)
        if result_df is not None:
            all_features.append(result_df)
        else:
            failed_subjects[participant_id] = error_msg

    if all_features:
        final_df = pd.concat(all_features, ignore_index=True)
        output_dir = os.path.dirname(EXCEL_FILE_NAME)
        if not os.path.exists(output_dir):
            os.makedirs(output_dir)
        output_path = os.path.join(output_dir, 'ABIDE2_all_aparc_features.csv')
        final_df.to_csv(output_path, index=False)
        successful_subjects = len(final_df['participant_id'].unique())
        print(f"Done. {successful_subjects} subjects, saved to {output_path}")
    else:
        print("All subjects failed; no output generated.")

    if failed_subjects:
        print("\n--- Failed subjects ---")
        for sub, reason in failed_subjects.items():
            print(f"{sub}: {reason}")